# מפגש 28 — חקר Titanic מלא | מעבדת תלמיד
עובדים לפי מקור התרגול: יעד Survived=1 חיובי; מאפיינים Age, Fare, SibSp, Sex, Pclass; חלוקה test_size=.25, seed17. מקבעים כלל בחירה לפני Test, משווים KNN, SVM וקו בסיס ב־CV על Train בלבד. השלימו `???`, כתבו פירושים בעצמכם. אין לשנות בחירה לאחר הערכת Test.

## 0. חישוב ידני
בדוגמה מומצאת אמת [0,0,1,1], חיזוי [0,1,1,1]. לכן TP=2, FP=1, FN=0. חשבו F1=2TP/(2TP+FP+FN).

In [ ]:
from sklearn.metrics import f1_score
toy_true=[0,0,1,1];toy_pred=[0,1,1,1]
toy_f1=2*???/(2*???+???+???)
assert abs(toy_f1-f1_score(toy_true,toy_pred))<1e-12
print("Synthetic F1",toy_f1)

## 1. מקור ושאלה
טענו נתונים ושמרו על ייחוד PassengerId. העמודה Survived אינה מאפיין. אין להשלים תוויות יעד חסרות.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split,StratifiedKFold,GridSearchCV,cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score,recall_score,f1_score
url="https://raw.githubusercontent.com/arielb30/datasets/main/titanic_data.csv"
raw=pd.read_csv(url)
assert "PassengerId" in raw.columns
assert raw["PassengerId"].is_unique
print(raw.shape)

## 2. הגדרת מאפיינים ופיצול
עוצרים לפני EDA ובוחרים Train/Test. בחרו numeric עם Age, Fare, SibSp, ו־categorical עם Sex ו־Pclass. שיעור הישרדות אינו הוכחת סיבתיות.

In [ ]:
numeric=[???,???,???]
categorical=[???,???]
features=numeric+categorical
data=raw.dropna(subset=[???]).copy()
X=data[features].copy()
y=data[???].astype(int)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=???,random_state=???,stratify=y)
assert set(X_train.index).isdisjoint(set(X_test.index))
assert set(features).isdisjoint({"Survived","PassengerId"})
assert set(y.unique()).issubset({0,1})

## 3. EDA רק על Train
דווחו שיעור הישרדות לפי Pclass ומספר חסרים; הדגימו לפחות גרף אחד. אין לחשוף Test כדי לבחור מאפיינים.

In [ ]:
development=X_train.copy()
development["Survived"]=y_train
missing=development.???().sum()
class_rates=development.groupby(???)["Survived"].???()
print(missing);print(class_rates)
assert len(class_rates)>0
plt.figure()
sns.countplot(data=development,x="Survived")
plt.xlabel("Survived on Train")
plt.show()

## 4. הכנה ב־Pipeline ו־GridSearchCV
מגדירים מראש חמישה קיפולים, מועמדי K=[3,5,9] ומדד F1 לחיובי 1. כל imputation/encoding מתבצע בתוך הקיפול.

In [ ]:
num_pipe=Pipeline([("fill",SimpleImputer(strategy=???)),("scale",StandardScaler())])
cat_pipe=Pipeline([("fill",SimpleImputer(strategy=???)),("encode",OneHotEncoder(handle_unknown="ignore",sparse_output=False))])
prepare=ColumnTransformer([("numeric",num_pipe,numeric),("categorical",cat_pipe,categorical)])
knn=Pipeline([("prepare",prepare),("model",???())])
cv=StratifiedKFold(n_splits=???,shuffle=True,random_state=42)
grid={"model__n_neighbors":[3,5,9]}
clf=GridSearchCV(knn,param_grid=grid,cv=cv,scoring=???,refit=True,error_score="raise")
clf.fit(X_train,y_train)
results=pd.DataFrame(clf.cv_results_)
assert clf.best_params_["model__n_neighbors"] in [3,5,9]
display(results[["param_model__n_neighbors","mean_test_score","rank_test_score"]])

## 5. השוואת שלושה מודלים באותו CV
SVM ליניארי C=1 וקו בסיס most_frequent נשארים קבועים. כלל שוויון: KNN קודם, לאחריו SVM ולבסוף Baseline. אין לגלות Test בזמן בחירה.

In [ ]:
svm=Pipeline([("prepare",prepare),("model",SVC(kernel=???,C=???))])
baseline=DummyClassifier(strategy=???)
svm_scores=cross_val_score(svm,X_train,y_train,cv=cv,scoring=???)
base_scores=cross_val_score(baseline,X_train,y_train,cv=cv,scoring=???)
cv_table=pd.DataFrame({"model":["KNN","Linear SVM","Baseline"],"mean_cv_f1":[clf.best_score_,svm_scores.mean(),base_scores.mean()]})
chosen_name=cv_table.sort_values("mean_cv_f1",ascending=False,kind="stable").iloc[0]["model"]
chosen={"KNN":clf.best_estimator_,"Linear SVM":svm,"Baseline":baseline}[chosen_name]
chosen.fit(X_train,y_train)
baseline.fit(X_train,y_train)
assert chosen_name in cv_table["model"].values
print("Pre-Test choice",chosen_name)

## 6. Test סופי — חד פעמי
אפשר לחשב כמה מדדים מאותה תחזית. אין לעדכן את פרמטרי המודל לאחר חשיפת Test ולכנות את אותו Test מבחן סופי בלתי תלוי.

In [ ]:
predictions=chosen.predict(???)
baseline_predictions=baseline.predict(???)
matrix=confusion_matrix(???,???,labels=[0,1])
tn,fp,fn,tp=matrix.ravel()
f1_manual=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.
f1_lib=f1_score(???,???,zero_division=0)
assert np.isclose(f1_manual,f1_lib)
assert matrix.sum()==len(y_test)
print("Test F1",f1_lib,"Baseline F1",f1_score(y_test,baseline_predictions,zero_division=0))
display(pd.DataFrame(matrix,index=["Actual 0","Actual 1"],columns=["Predicted 0","Predicted 1"]))

## 7. מסקנות והגנה
כתבו שתי טעויות מן המטריצה, מגבלה של הכללה למשפחות חדשות והחלטה אחת שנקבעה לפני Test. אם Test מאכזב — מה צריך לתעד, ומדוע אין לשנות את המתכון ולדווח על אותו Test כאילו נשמר? הפרויקט האישי הוא התוצר; אין חובה לפרויקט Titanic נוסף.